# TRACT: From Zero-Shot to Expert-Reviewed Security Crosswalk

**A practitioner's guide to mapping AI security frameworks using machine learning**

This notebook tells the story of how we built a system that reads any security control and tells you which part of a universal security taxonomy it belongs to. We'll walk through every experiment, every failure, and every hard-won insight — in plain language.

---

In [ ]:
"""Setup: imports, seeds, prerequisite checks."""
import sys
import json
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from sklearn.manifold import TSNE

# Ensure nb_helpers is importable from notebooks/
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent))

from nb_helpers import (
    PROJECT_ROOT, RESULTS_DIR, DATA_DIR,
    PHASE0_DIR, PHASE1B_DIR, PHASE1C_DIR,
    REVIEW_DIR, BRIDGE_DIR, DATASET_DIR,
    OKABE_ITO, SEQUENTIAL_BLUE, SEQUENTIAL_ORANGE, DIVERGING,
    FigureCounter, style_axes, plotly_with_fallback,
    load_phase0_experiment, load_firewalled_baseline,
    load_fold_metrics, load_training_logs,
    load_calibration_data, load_deployment_embeddings,
    load_review_metrics, load_review_export,
    load_cre_hierarchy, load_crosswalk, load_framework_metadata,
    check_prerequisites,
)

# Reproducibility
random.seed(42)
np.random.seed(42)

# Matplotlib defaults
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "savefig.bbox": "tight",
})
sns.set_style("whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

# Figure counter — instantiate once
fig_counter = FigureCounter()

# Prerequisite check
missing = check_prerequisites()
if missing:
    print("⚠️  Missing prerequisite files:")
    for m in missing:
        print(f"   - {m}")
    print("\nSee the notebook README for setup instructions.")
else:
    print("✓ All prerequisite files found")

# Verify CWD
cwd = Path.cwd()
if cwd.name != "notebooks":
    print(f"⚠️  Expected CWD = notebooks/, got {cwd.name}/. Shell cells may not work correctly.")
else:
    print(f"✓ CWD: {cwd}")

print(f"✓ PROJECT_ROOT: {PROJECT_ROOT}")
print(f"✓ NumPy {np.__version__}, Matplotlib {matplotlib.__version__}")

## 1. Why This Exists: The Security Framework Translation Problem

You're a security architect. Your organization runs workloads across three cloud providers, 
deploys AI models in production, and needs to demonstrate compliance with half a dozen 
frameworks — NIST 800-53, ISO 27001, OWASP, MITRE ATLAS, the EU AI Act, and more.

Each framework has its own taxonomy, its own control numbering, its own way of saying 
"encrypt data at rest." Comparing them manually means reading thousands of controls 
and building mental bridges between different vocabularies for the same concepts.

This is the **N² problem**: with *k* frameworks, you need *k(k−1)/2* pairwise mappings.

### The Common Requirements Enumeration (CRE)

The [CRE project](https://opencre.org) solves this by creating a **shared coordinate system** 
for security requirements. Think of it as GPS coordinates for security concepts — instead of 
describing locations relative to each other ("the bakery is two blocks from the bank"), 
you assign each location a universal coordinate.

CRE organizes 522 security concept **hubs** into a hierarchy. Every security control from 
every framework can be mapped to one or more hubs. Once mapped, cross-framework comparison 
becomes a simple lookup: "which controls share the same hub?"

Let's look at this hierarchy:

In [ ]:
hierarchy = load_cre_hierarchy()
hubs = hierarchy["hubs"]
print(f"CRE hierarchy: {len(hubs)} hubs")

In [ ]:
ids, names, parents = [], [], []
for hub_id, hub in sorted(hubs.items()):
    ids.append(hub_id)
    names.append(hub["name"])
    parents.append(hub.get("parent_id") or "")

fig = go.Figure(go.Sunburst(
    ids=ids,
    labels=names,
    parents=parents,
    maxdepth=3,
))
fig_num = fig_counter.next(1)
plotly_with_fallback(fig, fig_num, "CRE Hub Taxonomy — 522 Hubs")

The sunburst above shows the full CRE taxonomy. Click to drill down into any branch. 
The outer rings show increasingly specific security concepts — from broad areas like 
"Authentication" and "Cryptography" down to specific requirements like 
"Password complexity" or "Key rotation."

This is our target space: every security control we encounter will be mapped to one of 
these 522 hubs.

### The Assignment Paradigm

TRACT uses an **assignment paradigm**: instead of comparing controls to each other, 
we train a model to map each control independently to the CRE hierarchy:

$$g(\text{control\_text}) \rightarrow \text{CRE\_hub}$$

This is fundamentally different from pairwise comparison. Here's why it matters:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: N² pairwise
n_fw = 6
for i in range(n_fw):
    for j in range(i+1, n_fw):
        ax1.plot([i, j], [0, 0], 'o-', color=OKABE_ITO[4], alpha=0.3, markersize=8)
ax1.set_xlim(-0.5, n_fw-0.5)
ax1.set_title(f"Pairwise: {n_fw*(n_fw-1)//2} comparisons", fontsize=12)
ax1.set_yticks([])

# Right: Assignment paradigm
for i in range(n_fw):
    ax2.annotate("", xy=(3, 2-i*0.7), xytext=(i, -2),
                 arrowprops=dict(arrowstyle="->", color=OKABE_ITO[i % len(OKABE_ITO)]))
ax2.set_title(f"Assignment: {n_fw} mappings", fontsize=12)
ax2.set_yticks([])

fig_num = fig_counter.next(1)
fig.suptitle(f"{fig_num}: Why Assignment Scales", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
plt.close(fig)

With 6 frameworks, pairwise comparison requires 15 separate mappings. Add a 7th framework 
and you need 21 — every new framework means comparing against *all* existing ones.

The assignment paradigm? Just 6 mappings (now 7). Each framework maps to the shared 
coordinate system independently. Cross-framework comparison is then a free side effect 
of the mapping — frameworks that map to the same hub are, by definition, addressing the 
same security concept.

### What This Notebook Covers

1. **Data Landscape** — What we're working with: 31 frameworks, 5,238 assignments, 522 hubs
2. **Phase 0: Zero-Shot Baselines** — How well do off-the-shelf models do?
3. **Model Selection** — Choosing the right base embedding model
4. **Contrastive Fine-Tuning** — Teaching the model what "same hub" means
5. **Ablation Analysis** — What actually helps (and what doesn't)
6. **Hub Firewall** — How we keep evaluation honest
7. **Final Results** — Aggregate performance with bootstrap confidence intervals
8. **Error Analysis** — Where the model struggles and why
9. **Calibration** — Can we trust the confidence scores?
10. **Human Review** — Expert validation of 5,238 assignments
11. **CLI Tutorial** — Using TRACT in practice
12. **Retrospective** — What we built, what we learned, what's next

> **Plain English:** We built a system that reads any security control and tells you 
> which part of a universal security taxonomy it belongs to. This lets you instantly 
> compare any two frameworks — without reading thousands of controls manually.

## 2. Data Landscape: What We're Working With

Before we can train anything, we need to understand our data. TRACT's crosswalk dataset 
contains assignments from 31 security frameworks to 522 CRE hubs — but the distribution 
is far from uniform.

In [ ]:
crosswalk = load_crosswalk()
fw_metadata = load_framework_metadata()
hierarchy = load_cre_hierarchy()
hubs = hierarchy["hubs"]

print(f"Crosswalk: {len(crosswalk)} assignments")
print(f"Frameworks: {len(fw_metadata)}")
print(f"Hubs: {len(hubs)}")

In [ ]:
def get_root_hub(hub_id: str) -> str:
    """Walk up the hierarchy to find the root ancestor."""
    current = hub_id
    while current in hubs and (hubs[current].get("parent_id") or None):
        current = hubs[current]["parent_id"]
    return current

# Count flows: framework_name → root_hub_name
flows = Counter()
for row in crosswalk:
    root = get_root_hub(row["hub_id"])
    root_name = hubs[root]["name"] if root in hubs else root
    flows[(row["framework_name"], root_name)] += 1

# Top 5 root hubs by total flow (avoid visual clutter)
root_totals = Counter()
for (fw, root), count in flows.items():
    root_totals[root] += count
top_roots = {r for r, _ in root_totals.most_common(5)}

# Build Sankey node/link arrays
fw_names = sorted({fw for fw, root in flows if root in top_roots})
root_names = sorted(top_roots)
labels = fw_names + root_names
label_idx = {name: i for i, name in enumerate(labels)}

sources, targets, values = [], [], []
for (fw, root), count in flows.items():
    if root in top_roots and fw in label_idx:
        sources.append(label_idx[fw])
        targets.append(label_idx[root])
        values.append(count)

colors = [OKABE_ITO[i % len(OKABE_ITO)] for i in range(len(labels))]

fig = go.Figure(go.Sankey(
    node=dict(label=labels, color=colors, pad=15),
    link=dict(source=sources, target=targets, value=values),
))
fig_num = fig_counter.next(2)
plotly_with_fallback(fig, fig_num, "Framework-to-Hub Cluster Mappings")

The Sankey diagram shows how controls from different frameworks flow into the top CRE 
hub clusters. Notice how some frameworks (like NIST 800-53) spread broadly across many 
security domains, while others (like OWASP Top 10 for LLM) concentrate in specific areas.

This flow pattern is exactly what makes cross-framework comparison valuable — and challenging.

In [ ]:
hub_link_counts = Counter(row["hub_id"] for row in crosswalk)
counts = sorted(hub_link_counts.values(), reverse=True)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(counts)), counts, color=OKABE_ITO[4], alpha=0.8)
# Annotate 80/20 point
cumsum = np.cumsum(counts) / sum(counts)
p80 = np.searchsorted(cumsum, 0.8)
ax.axvline(p80, color=OKABE_ITO[5], linestyle="--", label=f"80% of links in top {p80} hubs")
ax.legend()
fig_num = fig_counter.next(2)
style_axes(ax, "Hub Link Distribution — The Long Tail", "Hubs (sorted by link count)", "Number of links", fig_num)
plt.tight_layout()
plt.show()
plt.close(fig)

The hub link distribution follows a classic long tail: a small number of hubs attract 
many links, while most hubs have only a few. This has practical implications for our model — 
hubs with many training examples will be easier to learn than rare ones.

In [ ]:
fw_names_list = [fw["framework_name"] for fw in fw_metadata]
fw_counts = [fw["total_controls"] for fw in fw_metadata]
fw_types = [fw["coverage_type"] for fw in fw_metadata]
colors_bar = [OKABE_ITO[2] if t == "ground_truth" else OKABE_ITO[0] for t in fw_types]

fig, ax = plt.subplots(figsize=(10, 8))
y_pos = range(len(fw_names_list))
ax.barh(y_pos, fw_counts, color=colors_bar)
ax.set_yticks(y_pos)
ax.set_yticklabels(fw_names_list, fontsize=9)
fig_num = fig_counter.next(2)
style_axes(ax, "Controls per Framework", "Number of controls", "", fig_num)
plt.tight_layout()
plt.show()
plt.close(fig)

### Multi-Hub Mappings

Not every control maps to exactly one CRE hub. Some security controls are broad enough 
to span multiple concepts. Let's see how common this is:

In [ ]:
from collections import defaultdict
control_hubs = defaultdict(set)
for row in crosswalk:
    control_hubs[row["control_id"]].add(row["hub_id"])

hub_counts = [len(h) for h in control_hubs.values()]
multi = sum(1 for c in hub_counts if c > 1)
total = len(hub_counts)
print(f"Controls mapping to >1 hub: {multi}/{total} ({100*multi/total:.1f}%)")
print(f"Max hubs per control: {max(hub_counts)}")
print(f"Mean hubs per control: {np.mean(hub_counts):.2f}")

Multi-hub mappings are common and correct — a control like "implement access control 
for AI model training data" genuinely touches both access control and AI-specific concerns. 
Our evaluation metrics (hit@1, hit@5) account for this: a prediction is correct if it 
matches *any* of the control's ground-truth hubs.

### Data Provenance

The training data comes from two sources:
- **OpenCRE LinkedTo** — Expert-curated links from the CRE project team
- **OpenCRE AutomaticallyLinkedTo** — Deterministic CAPEC→CWE→CRE transitive chains 
  (not ML output — these are as reliable as expert links)

Both are treated equally during training and evaluation.

> **Plain English:** We have 5,238 assignments across 31 frameworks, but the distribution 
> is uneven — some hubs get lots of training data, others very little. About 35% of 
> controls map to multiple hubs, which our evaluation handles correctly.

## 3. Phase 0: Can Off-the-Shelf Models Do This?

Before investing in fine-tuning, we need to know: how well do existing models handle
security control assignment? We tested seven approaches — from classical NLI to
state-of-the-art LLMs — to establish baselines and identify the most promising direction.

In [ ]:
bge = load_phase0_experiment("exp1_embedding_baseline_bge")
gte = load_phase0_experiment("exp1_embedding_baseline_gte")
deberta = load_phase0_experiment("exp1_embedding_baseline_deberta")
opus = load_phase0_experiment("exp2_llm_probe")
knn = load_phase0_experiment("exp5_knn_baseline")
fewshot = load_phase0_experiment("exp6_fewshot_sonnet")

models = {
    "DeBERTa-v3-NLI": deberta["models"]["deberta-v3-nli"]["all_198"]["hit_at_1"]["mean"],
    "kNN (k=5)": knn["k_values"]["k5"]["all"]["hit_at_1"]["mean"],
    "GTE-large": gte["models"]["gte-large-v1.5"]["all_198"]["hit_at_1"]["mean"],
    "BGE-large-v1.5": bge["models"]["bge-large-v1.5"]["all_198"]["hit_at_1"]["mean"],
    "Few-shot Sonnet 4\n(desc)": fewshot["variants"]["sonnet-desc"]["all"]["hit_at_1"]["mean"],
    "Few-shot Sonnet 4\n(no desc)": fewshot["variants"]["sonnet-nodesc"]["all"]["hit_at_1"]["mean"],
    "Claude Opus 4": opus["all_198"]["hit_at_1"]["mean"],
}

print("Zero-shot baselines (hit@1, unfirewalled):")
for name, score in sorted(models.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name.replace(chr(10), ' ')}: {score:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
names = list(models.keys())
scores = list(models.values())
colors = [OKABE_ITO[5] if s == 0 else OKABE_ITO[4] if s < 0.3 else OKABE_ITO[2] for s in scores]
bars = ax.bar(range(len(names)), scores, color=colors)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=9, ha="center")
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{score:.3f}", ha="center", va="bottom", fontsize=9)
fig_num = fig_counter.next(3)
style_axes(ax, "Zero-Shot Baselines — Hit@1 (Unfirewalled)", "", "Hit@1", fig_num)
plt.tight_layout()
plt.show()
plt.close(fig)

### The DeBERTa Disaster

The first thing that jumps out: **DeBERTa-v3-NLI scores exactly 0.000.** This isn't a bug —
it's a fundamental mismatch. DeBERTa-v3-NLI was trained for Natural Language Inference
(entailment/contradiction/neutral), not semantic similarity. It produces a 3-class
probability distribution, not an embedding vector you can compare with cosine similarity.

This killed an entire class of approaches: **NLI models are not suitable for hub assignment.**
We never looked back.

### BGE-large Wins Zero-Shot

Among embedding models, BGE-large-v1.5 leads with hit@1 ≈ 0.348 — meaning about 1 in 3
controls are assigned to the correct hub on the first try, with no training at all. Not
great, but it proves the embedding space has signal we can amplify with fine-tuning.

GTE-large (hit@1 ≈ 0.267) trails behind and shows more variance across frameworks.

In [ ]:
# Per-fold data for BGE, GTE
bge_folds = bge["models"]["bge-large-v1.5"]["per_fold"]
gte_folds = gte["models"]["gte-large-v1.5"]["per_fold"]

# Build per-framework comparison
fw_names_phase0 = [f["framework"] for f in bge_folds]
bge_scores = [f["metrics"]["hit_at_1"] for f in bge_folds]
gte_scores = [f["metrics"]["hit_at_1"] for f in gte_folds]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(fw_names_phase0))
w = 0.35
ax.bar(x - w/2, bge_scores, w, label="BGE-large-v1.5", color=OKABE_ITO[4])
ax.bar(x + w/2, gte_scores, w, label="GTE-large", color=OKABE_ITO[0])
ax.set_xticks(x)
ax.set_xticklabels(fw_names_phase0, rotation=45, ha="right", fontsize=8)
ax.legend()
fig_num = fig_counter.next(3)
style_axes(ax, "Per-Framework Zero-Shot Performance", "", "Hit@1", fig_num)
plt.tight_layout()
plt.show()
plt.close(fig)

### The Opus Ceiling

Claude Opus 4 (`claude-opus-4-20250514`), used as a zero-shot classifier with full hub
descriptions, achieves hit@1 ≈ 0.553 (unfirewalled, n=197). This represents the upper
bound for what a general-purpose LLM can achieve without any task-specific training.

At approximately $0.60 per control, it's 1,000× more expensive than our embedding approach
at inference time. The question becomes: can fine-tuning close the gap at a fraction of the cost?

**Important:** This 0.553 is unfirewalled — the hub representations include information from
the same frameworks being evaluated. Our fine-tuned model's 0.537 is firewalled (LOFO),
making direct comparison inappropriate. We'll address this honestly in Section 8.

In [ ]:
paths_bge = load_phase0_experiment("exp3_hierarchy_paths_bge")
descs = load_phase0_experiment("exp4_hub_descriptions")

paths_delta = paths_bge["models"]["bge-large-v1.5"]["deltas_all_198"]["hit_at_1"]
delta_paths = paths_delta["delta_mean"]
ci_low_paths = paths_delta["ci_low"]
ci_high_paths = paths_delta["ci_high"]

descs_delta = descs["models"]["bge-large-v1.5"]["deltas_subset"]["hit_at_1"]
delta_descs = descs_delta["delta_mean"]
ci_low_descs = descs_delta["ci_low"]
ci_high_descs = descs_delta["ci_high"]

print(f"Hierarchy paths: Δhit@1 = {delta_paths:+.3f} (95% CI: [{ci_low_paths:.3f}, {ci_high_paths:.3f}])")
print(f"Hub descriptions: Δhit@1 = {delta_descs:+.3f} (95% CI: [{ci_low_descs:.3f}, {ci_high_descs:.3f}])")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
factors = [
    ("Hierarchy paths", delta_paths, ci_low_paths, ci_high_paths),
    ("Hub descriptions", delta_descs, ci_low_descs, ci_high_descs),
]
for i, (name, delta, lo, hi) in enumerate(factors):
    color = OKABE_ITO[2] if delta > 0 else OKABE_ITO[5]
    ax.errorbar(delta, i, xerr=[[delta - lo], [hi - delta]], fmt='o',
                color=color, capsize=5, markersize=10, linewidth=2)
    ax.text(delta + 0.01 if delta > 0 else delta - 0.01, i + 0.15,
            f"{delta:+.3f}", ha="left" if delta > 0 else "right", fontsize=10)
ax.axvline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_yticks(range(len(factors)))
ax.set_yticklabels([f[0] for f in factors])
fig_num = fig_counter.next(3)
style_axes(ax, "Zero-Shot Ablation: Feature Impact on Hit@1", "Δ Hit@1", "", fig_num)
plt.tight_layout()
plt.show()
plt.close(fig)

### Structure Beats Prose

Hierarchy paths (+7.6%, CI: [1.5%, 13.6%]) significantly improve performance. Adding the
path "Authentication > Multi-Factor > Hardware Tokens" to each hub gives the embedding model
structural context it can't infer from names alone.

Hub descriptions (-4.8%, CI: [-12.4%, 2.8%]) actually *hurt* performance. The CI crosses
zero, so the effect may be null rather than negative — but it's certainly not helpful.
Long prose descriptions add noise that drowns out the structural signal.

**Decision: Use hierarchy paths, skip descriptions.** This shaped our entire fine-tuning approach.

This is a pattern we see throughout TRACT: **structured, hierarchical information outperforms
unstructured text** for this specific task. Security taxonomies are inherently structured —
the model needs to learn *position in a tree*, not *similarity of paragraphs*.

### Key Takeaway

BGE-large-v1.5 with hierarchy paths gives us a 0.348 baseline that's clearly trainable.
The embedding space has real signal — controls about the same security concept land in
similar neighborhoods. Fine-tuning should amplify this signal dramatically.

**⚠️ Small-fold caveat:** Some LOFO folds have very few test controls (e.g., OWASP Top 10
for LLM: n=6, OWASP Top 10 for ML: n=7). Per-fold metrics for these frameworks have wide
confidence intervals and should be interpreted with caution. We report them for completeness
but draw conclusions primarily from the aggregate metrics and larger folds.

> **Plain English:** We tried seven approaches. NLI models don't work at all. LLMs work
> well but cost too much. BGE-large embedding model gets 1-in-3 right out of the box, and
> adding hierarchy path structure improves it by 7.6%. That's our starting point for fine-tuning.

## 4. Base Model Selection: Per-Framework Complementarity

BGE-large-v1.5 won the aggregate comparison, but the aggregate hides important per-framework 
variation. Before committing to a single model, we need to understand where each model 
excels — and whether a different model might be better for specific framework types.

In [ ]:
# Build model × framework performance matrix from Phase 0
bge_folds = bge["models"]["bge-large-v1.5"]["per_fold"]
gte_folds = gte["models"]["gte-large-v1.5"]["per_fold"]

frameworks = [f["framework"] for f in bge_folds]
perf_matrix = {
    "BGE-large-v1.5": [f["metrics"]["hit_at_1"] for f in bge_folds],
    "GTE-large": [f["metrics"]["hit_at_1"] for f in gte_folds],
}
print(f"Performance matrix: {len(perf_matrix)} models × {len(frameworks)} frameworks")

In [ ]:
import pandas as pd

df = pd.DataFrame(perf_matrix, index=frameworks).T
fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(df.values, cmap=SEQUENTIAL_BLUE, aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(df.columns)))
ax.set_xticklabels(df.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(df.index)))
ax.set_yticklabels(df.index, fontsize=10)
for i in range(len(df.index)):
    for j in range(len(df.columns)):
        val = df.values[i, j]
        color = "white" if val > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=color)
plt.colorbar(im, ax=ax, label="Hit@1", shrink=0.8)
fig_num = fig_counter.next(4)
style_axes(ax, "Zero-Shot Hit@1 by Model × Framework", "", "", fig_num)
plt.tight_layout()
plt.show()
plt.close(fig)

### Selection Rationale

BGE-large-v1.5 isn't just the best on average — it's the most **consistent**. While GTE-large 
occasionally matches or exceeds BGE on individual frameworks, its performance is more 
variable. For a model we'll fine-tune across all frameworks simultaneously, consistency 
matters more than peak performance on any single fold.

**Decision: BGE-large-v1.5 as the base model for contrastive fine-tuning.**

In [ ]:
# Use BASE BGE embeddings — Section 4 is about model selection, BEFORE fine-tuning
base_path = RESULTS_DIR / "phase1b" / "base_bge_embeddings.npz"
base_data = np.load(str(base_path), allow_pickle=False)
ctrl_emb_base = base_data["control_embeddings"]
ctrl_ids_base = base_data["control_ids"]

# Subsample to ~500 controls for Plotly performance
rng = np.random.RandomState(42)
n_sample = min(500, len(ctrl_emb_base))
idx = rng.choice(len(ctrl_emb_base), n_sample, replace=False)
sample_emb = ctrl_emb_base[idx]
sample_ids = ctrl_ids_base[idx]

# Extract framework prefix from control_ids (format: framework::control_id)
sample_frameworks = [str(cid).split("::")[0] if "::" in str(cid) else "unknown" for cid in sample_ids]

# t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
coords = tsne.fit_transform(sample_emb)
print(f"t-SNE complete: {coords.shape[0]} points")

In [ ]:
# Assign colors by framework
unique_fws = sorted(set(sample_frameworks))
fw_color_map = {fw: OKABE_ITO[i % len(OKABE_ITO)] for i, fw in enumerate(unique_fws)}

fig = go.Figure()
for fw in unique_fws:
    mask = [f == fw for f in sample_frameworks]
    fw_coords = coords[mask]
    fw_ids_filtered = [str(sample_ids[i]) for i, m in enumerate(mask) if m]
    fig.add_trace(go.Scatter(
        x=fw_coords[:, 0], y=fw_coords[:, 1],
        mode="markers",
        name=fw,
        marker=dict(size=5, color=fw_color_map[fw], opacity=0.7),
        text=fw_ids_filtered,
        hovertemplate="%{text}<extra>%{fullData.name}</extra>",
    ))
fig_num = fig_counter.next(4)
plotly_with_fallback(fig, fig_num, "Base BGE Embedding Space (Pre-Fine-Tuning, t-SNE)")

**⚠️ t-SNE caveat:** Distances between clusters in t-SNE are *not meaningful* — only the 
existence of clusters matters. Two clusters that appear far apart may be closer in the 
original 1024-dimensional space than two clusters that appear near each other. Perplexity=30, 
random_state=42.

### What the Clusters Reveal

Even before fine-tuning, the base BGE model groups controls from similar frameworks 
together. You can see distinct clusters for different security domains. This confirms 
that the pre-trained embedding space already captures meaningful security concept 
relationships — our fine-tuning will sharpen these boundaries.

### Model Selection Summary

| Criterion | BGE-large-v1.5 | GTE-large |
|-----------|----------------|-----------||
| Aggregate hit@1 | Higher | Lower |
| Consistency | More consistent | More variable |
| Embedding dim | 1024 | 1024 |
| Parameters | 335M | 335M |
| With hierarchy paths | +7.6% boost | Not tested |

BGE-large-v1.5 provides the strongest and most consistent foundation for fine-tuning.

> **Plain English:** We compared embedding models head-to-head across every framework. 
> BGE-large-v1.5 won not just on average, but by being the most consistent performer — 
> important when you're fine-tuning one model for all frameworks simultaneously.